# TripMe Part 5 — QLoRA Smoke Training
Attach `tripme-part05-smoke`, enable a T4 GPU, and Run All. This verifies the pipeline; it is not the final model.

In [ ]:
# Kaggle already provides datasets/fsspec/dill. Do not downgrade them, because that conflicts with gcsfs/TPOT.
!pip install -q --no-cache-dir transformers==4.48.3 accelerate==1.3.0 peft==0.14.0 bitsandbytes==0.48.2 safetensors>=0.4
# Kaggle may expose T4 x2. QLoRA smoke training intentionally uses one GPU;
# torch DataParallel is unsafe for a model quantized and pinned to device 0.
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
import torch, bitsandbytes as bnb
print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda, 'bitsandbytes:', bnb.__version__)
assert torch.cuda.is_available(), 'GPU is not enabled'
assert torch.cuda.device_count() == 1, f'Expected one visible GPU, found {torch.cuda.device_count()}'


In [ ]:
from pathlib import Path
import json, random, shutil
import torch
assert torch.cuda.is_available(), 'Enable a Kaggle GPU before running'
MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'
MAX_LENGTH = 768
TRAIN_ROWS = 200
VAL_ROWS = 40
SEED = 42
matches = list(Path('/kaggle/input').rglob('pilot_train_draft.jsonl'))
if not matches: raise FileNotFoundError('Attach the tripme-part05-smoke dataset')
DATA_DIR = matches[0].parent
OUTPUT_DIR = Path('/kaggle/working/tripme-smoke-adapter')
print('GPU:', torch.cuda.get_device_name(0)); print('Data:', DATA_DIR)


In [ ]:
from datasets import load_dataset
data = load_dataset('json', data_files={'train': str(DATA_DIR/'pilot_train_draft.jsonl'), 'validation': str(DATA_DIR/'pilot_validation_draft.jsonl')})
data['train'] = data['train'].shuffle(seed=SEED).select(range(min(TRAIN_ROWS, len(data['train']))))
data['validation'] = data['validation'].shuffle(seed=SEED).select(range(min(VAL_ROWS, len(data['validation']))))
print(data)


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token; tokenizer.padding_side = 'right'
quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=quant, device_map={'': 0}, torch_dtype=torch.float16)
model = prepare_model_for_kbit_training(model); model.config.use_cache = False
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM', target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])
model = get_peft_model(model, lora); model.print_trainable_parameters()


In [ ]:
def tokenize_completion(example):
    full_text = tokenizer.apply_chat_template(example['messages'], tokenize=False, add_generation_prompt=False)
    prompt_text = tokenizer.apply_chat_template(example['messages'][:-1], tokenize=False, add_generation_prompt=True)
    full = tokenizer(full_text, truncation=True, max_length=MAX_LENGTH, add_special_tokens=False)
    prompt = tokenizer(prompt_text, truncation=True, max_length=MAX_LENGTH, add_special_tokens=False)
    labels = full['input_ids'].copy()
    labels[:min(len(prompt['input_ids']), len(labels))] = [-100] * min(len(prompt['input_ids']), len(labels))
    full['labels'] = labels
    return full
tokenized = data.map(tokenize_completion, remove_columns=data['train'].column_names)
assert any(value != -100 for value in tokenized['train'][0]['labels']), 'Assistant labels were fully masked'


In [ ]:
from transformers import DataCollatorForSeq2Seq, Trainer, TrainingArguments
args = TrainingArguments(output_dir=str(OUTPUT_DIR), num_train_epochs=1, per_device_train_batch_size=1, per_device_eval_batch_size=1, gradient_accumulation_steps=8, learning_rate=1e-4, logging_steps=5, eval_strategy='steps', eval_steps=25, save_strategy='steps', save_steps=25, save_total_limit=1, fp16=True, gradient_checkpointing=True, gradient_checkpointing_kwargs={'use_reentrant': False}, optim='paged_adamw_8bit', report_to='none', seed=SEED)
collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True, label_pad_token_id=-100, return_tensors='pt')
trainer = Trainer(model=model, args=args, train_dataset=tokenized['train'], eval_dataset=tokenized['validation'], data_collator=collator)
result = trainer.train(); evaluation = trainer.evaluate()
trainer.save_model(str(OUTPUT_DIR)); tokenizer.save_pretrained(str(OUTPUT_DIR))


In [ ]:
metrics = {**result.metrics, **{f'final_{k}': v for k,v in evaluation.items()}}
(OUTPUT_DIR/'training_metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
manifest = {'status':'smoke_complete','model_id':MODEL_ID,'train_rows':len(data['train']),'validation_rows':len(data['validation']),'max_length':MAX_LENGTH,'seed':SEED,'dataset_status':'ai_draft_requires_human_review'}
(OUTPUT_DIR/'smoke_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
archive = shutil.make_archive('/kaggle/working/tripme_part05_smoke_output','zip',OUTPUT_DIR)
print(json.dumps({'manifest':manifest,'metrics':metrics},indent=2)); print('Download:',archive)


In [ ]:
from IPython.display import HTML, display
zip_path = '/kaggle/working/tripme_part05_smoke_output.zip'
assert Path(zip_path).is_file(), f'Missing output ZIP: {zip_path}'
print(f'ZIP size: {Path(zip_path).stat().st_size / 1024 / 1024:.1f} MB')
display(HTML("<a href='files/tripme_part05_smoke_output.zip' download>Download tripme_part05_smoke_output.zip</a>"))
